# A Deep-Genetic Algorithm (Deep-GA) Approach for High-Dimensional Nonlinear Parabolic Partial Differential Equations

## For Google Colaboratory and Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0,'/content/drive/MyDrive/Colab Notebooks/DeepGA')

!pip install munch

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Import Library

In [2]:
import os
import time
import json
import munch
import logging

import numpy as np
import pandas as pd
import tensorflow as tf

import equation as eqn
from modelDeepGA import BSDESolver

## Parameters for the Genetic Algorithm

In [3]:
start_time = time.time()

# minimum and maximum values of u_0
min_value = 0
max_value = 100
range_value = max_value - min_value

n = 10 # number of chromosomes in population
p_c = 0.8 # probability of crossover
p_m = 0.4 # probability of mutation
g = 15 # number of generations
p = 100 # number of training in each generation

## Parameters for BSDESolver

In [4]:
# The path to load json file
a = '/content/drive/MyDrive/Colab Notebooks/DeepGA/pricing_default_risk_d100.json'

with open(a) as json_data_file:
    config = json.load(json_data_file)
config = munch.munchify(config)
bsde = getattr(eqn, config.eqn_config.eqn_name)(config.eqn_config)
tf.keras.backend.set_floatx(config.net_config.dtype)

y_init = range_value/2
config.net_config.y_init_range = [y_init, y_init]

model = BSDESolver(config, bsde)
valid_data = model.get_valid_data()
loss = model.loss_ga(valid_data)

model.show_model()

Model: "nonshared_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 feed_forward_sub_net (FeedF  multiple                 35880     
 orwardSubNet)                                                   
                                                                 
 feed_forward_sub_net_1 (Fee  multiple                 35880     
 dForwardSubNet)                                                 
                                                                 
 feed_forward_sub_net_2 (Fee  multiple                 35880     
 dForwardSubNet)                                                 
                                                                 
 feed_forward_sub_net_3 (Fee  multiple                 35880     
 dForwardSubNet)                                                 
                                                                 
 feed_forward_sub_net_4 (Fee  multiple             

## The Genetic Algorithm

In [5]:
def createPopulation():
    a = range(1, n+1)
    b = np.array(a)
    c = b * range_value / n
    pop = c
    pop = pd.DataFrame(pop)
    pop.columns = ['u_0']
    
    return pop

def trainModel(p, popAll1, popAll2, popAll3):
    a = (np.sum(popAll1['u_0'])+np.sum(popAll1['u_0'])+np.sum(popAll1['u_0']))/(len(popAll1)+len(popAll2)+len(popAll3))
    y_init = [a]
    model.set_y_init(y_init)
    model.train(p)

def fitness(pop, valid_data):
    fitness = []

    for i in range(len(pop)):
        a = pop['u_0'][i]
        y_init = [a]
        model.set_y_init(y_init)
        loss = model.loss_ga(valid_data)
        fitness.append(loss)
    
    pop['fitness'] = fitness
    
    return pop

def randomSelection():
    position = np.random.permutation(n)
    
    return position[0], position[1]

def crossover(pop):
    popc = []
    
    for i in range(len(pop)):
        if np.random.rand() <= p_c:
            a, b = randomSelection()
            popc.append((pop.loc[a][0] + pop.loc[b][0])/2)
            
    popc = pd.DataFrame(popc)
    popc.columns = ['u_0']
    
    return popc

def mutation(pop):
    popm = []
    
    for i in range(len(pop)):
        if np.random.rand() <= p_m:
            popm.append(pop.loc[i][0] + np.random.rand()*(max_value/50) - (max_value/100))
    
    if len(popm) == 0:
        popm.append(pop.loc[i][0] + np.random.rand()*(max_value/50) - (max_value/100))
    
    popm = pd.DataFrame(popm)
    popm.columns = ['u_0']
    
    return popm

def combinePopulation(pop1, pop2, pop3):
    popAll = pop1.copy()
    popAll = popAll.append(pop2)
    popAll = popAll.append(pop3)
    popAll = popAll.drop_duplicates()
    popAll.index = range(len(popAll))

    return popAll

def sort(pop):
    pop = pop.sort_values(by=['fitness'])
    pop.index = range(len(pop))

    return pop

def elimination(pop):
    pop = pop.head(n)
    
    return pop

def printU(pop):
    for i in range(len(pop)):
        print(pop['u_0'].iloc[i])

def printFitness(pop):
    for i in range(len(pop)):
        print(pop['fitness'].iloc[i])

def GA():
    pop1 = createPopulation()
    pop2 = createPopulation()
    pop3 = createPopulation()
    
    print()
    print('===================================================================')
    print('Generation 0')
    print()

    print('u_0 :')
    printU(pop1)
    printU(pop2)
    printU(pop3)
    print()

    print('Average u_0 :')
    print(str(np.average(pop1['u_0'])))
    print(str(np.average(pop2['u_0'])))
    print(str(np.average(pop3['u_0'])))
    print()

    print('Average u_0 in all populations : ' + str((np.average(pop1['u_0'])+np.average(pop2['u_0'])+np.average(pop3['u_0']))/3))
    elapsed_time = time.time() - start_time
    print('Total time : ' + str(elapsed_time))
    print('===================================================================')
    print()
    
    for i in range(0, g):
        popc1 = crossover(pop1)
        popc2 = crossover(pop2)
        popc3 = crossover(pop3)
        
        popm1 = mutation(pop1)
        popm2 = mutation(pop2)
        popm3 = mutation(pop3)
        
        popAll1 = combinePopulation(pop1, popc1, popm1)
        popAll2 = combinePopulation(pop2, popc2, popm2)
        popAll3 = combinePopulation(pop3, popc3, popm3)
        
        f = 1.0/100
        popAll1.loc[len(popAll1.index)] = np.average(pop1['u_0'])+range_value*f
        popAll1.loc[len(popAll1.index)] = np.average(pop1['u_0'])-range_value*f
        popAll2.loc[len(popAll2.index)] = np.average(pop2['u_0'])+range_value*f
        popAll2.loc[len(popAll2.index)] = np.average(pop2['u_0'])-range_value*f
        popAll3.loc[len(popAll3.index)] = np.average(pop3['u_0'])+range_value*f
        popAll3.loc[len(popAll3.index)] = np.average(pop3['u_0'])-range_value*f
        
        f = 2.0/100
        popAll1.loc[len(popAll1.index)] = np.average(pop1['u_0'])+range_value*f
        popAll1.loc[len(popAll1.index)] = np.average(pop1['u_0'])-range_value*f
        popAll2.loc[len(popAll2.index)] = np.average(pop2['u_0'])+range_value*f
        popAll2.loc[len(popAll2.index)] = np.average(pop2['u_0'])-range_value*f
        popAll3.loc[len(popAll3.index)] = np.average(pop3['u_0'])+range_value*f
        popAll3.loc[len(popAll3.index)] = np.average(pop3['u_0'])-range_value*f
        
        # buat train dengan input semua populasi
        trainModel(p, popAll1, popAll2, popAll3)
        
        valid_data = model.get_valid_data()
        popAll1 = fitness(popAll1, valid_data)
        popAll2 = fitness(popAll2, valid_data)
        popAll3 = fitness(popAll3, valid_data)
        
        popAll1 = sort(popAll1)
        popAll2 = sort(popAll2)
        popAll3 = sort(popAll3)
        
        pop1 = elimination(popAll1)
        pop2 = elimination(popAll2)
        pop3 = elimination(popAll3)
        
        print()
        print('===================================================================')
        print('Generation ' + str(i+1))
        print()

        print('u_0 :')
        printU(pop1)
        printU(pop2)
        printU(pop3)
        print()

        print('fitness :')
        printFitness(pop1)
        printFitness(pop2)
        printFitness(pop3)
        print()

        pop1.drop(['fitness'], axis=1)
        pop2.drop(['fitness'], axis=1)
        pop3.drop(['fitness'], axis=1)
        
        print('Average u_0 :')
        print(str(np.average(pop1['u_0'])))
        print(str(np.average(pop2['u_0'])))
        print(str(np.average(pop3['u_0'])))
        print()

        print('Average u_0 in all populations : ' + str((np.average(pop1['u_0'])+np.average(pop2['u_0'])+np.average(pop3['u_0']))/3))
        print('Average fitness in all populations : ' + str((np.average(pop1['fitness'])+np.average(pop2['fitness'])+np.average(pop3['fitness']))/3))
        elapsed_time = time.time() - start_time
        print('Total time : ' + str(elapsed_time))
        print('===================================================================')
        print()
    
    pop = combinePopulation(pop1, pop2, pop3)
    
    return pop

## Result

In [6]:
pop = GA()


Generation 0

u_0 :
10.0
20.0
30.0
40.0
50.0
60.0
70.0
80.0
90.0
100.0
10.0
20.0
30.0
40.0
50.0
60.0
70.0
80.0
90.0
100.0
10.0
20.0
30.0
40.0
50.0
60.0
70.0
80.0
90.0
100.0

Average u_0 :
55.0
55.0
55.0

Average u_0 in all populations : 55.0
Total time : 3.5403788089752197


Generation 1

u_0 :
80.0
70.0
65.0
89.36229818016623
90.0
60.0
57.0
56.0
55.0
54.0
75.0
80.0
70.0
65.0
90.0
60.0
59.669352865080285
57.0
56.0
54.0
80.0
80.63519849683857
70.0
90.0
60.0
57.0
56.0
55.0
54.0
53.0

fitness :
22.115074483905918
66.19306339563659
149.21254076161912
172.04324178627002
188.98166007087545
268.51878294883755
360.18055177730787
394.2460591141955
430.1151759796343
467.81731041092854
17.788964471772047
22.115074483905918
66.19306339563659
149.21254076161912
188.98166007087545
268.51878294883755
277.86169771962676
360.18055177730787
394.2460591141955
467.81731041092854
22.115074483905918
26.44015229342284
66.19306339563659
188.98166007087545
268.51878294883755
360.18055177730787
394.24605911419